In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

In [ ]:
tree_params = {"command": "uv", "args": ["run", "tree_server.py"]}
ports_params = {"command": "uv", "args": ["run", "ports_server.py"]}

async with MCPServerStdio(params=ports_params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

In [ ]:
instructions = """You are a project assistant. You can inspect the project directory 
structure and manage ports on the system. Use the tree tool to explore directories 
and the port tools to list, check, scan, and kill ports as needed."""

request = "Show me the project directory tree and list 5 currently listening ports between 1000 and 10000"

model = "gpt-4.1-mini"

In [ ]:
async with MCPServerStdio(params=tree_params, client_session_timeout_seconds=30) as tree_server:
    async with MCPServerStdio(params=ports_params, client_session_timeout_seconds=30) as ports_server:
        agent = Agent(
            name="system_assistant",
            instructions=instructions,
            model=model,
            mcp_servers=[tree_server, ports_server],
        )
        with trace("system_assistant"):
            result = await Runner.run(agent, request)
        display(Markdown(result.final_output))